# Data Collection: LLM Phishing Bias Evaluation

This notebook collects persona generation and phishing vulnerability assessment data
from 15 open-source LLMs across 5 providers, following the DECODINGTRUST framework
methodology adapted for phishing susceptibility (Week 5 lecture).

**Workflow per run:**
1. **Prompt 1** — LLM generates 3 diverse personas with demographics
2. **Prompt 2** — LLM selects which persona is most vulnerable to phishing and explains why
3. **Parse** — Extract structured fields (name, age, gender, etc.) via LLM-based extraction

**Scale:** 15 models × 20 runs = 300 workflows → 900 persona rows

**Research Question:** Do LLMs exhibit systematic bias (gender, age, education, occupation)
when assessing phishing susceptibility across diverse personas?

**References:**
- Wang et al. (2023). DECODINGTRUST: A Comprehensive Assessment of Trustworthiness in GPT Models.
- Butavicius et al. (2016). Age and personality traits as predictors of phishing susceptibility.
- Sheng et al. (2010). Age, gender, and education effects on phishing susceptibility.

## 1. Setup and Imports

In [1]:
# !pip install openai pandas tqdm

import sys
from pathlib import Path
if str(Path('..').resolve()) not in sys.path:
    sys.path.insert(0, str(Path('..').resolve()))

import json
import time
import os
import re
from datetime import datetime

import pandas as pd
from openai import OpenAI
from tqdm.notebook import tqdm

from src.config import API_KEYS, PROVIDERS, GENERATION_PARAMS, RUNS_PER_MODEL
from src.config import PARSE_PROVIDER, PARSE_MODEL, PARSE_PARAMS, REQUEST_DELAY
from src.prompts import PROMPT_1, PROMPT_2, PARSE_PROMPT

# Resolve canonical paths (relative to repo root)
RESULTS_DIR = Path('..') / 'results'
DATA_DIR = Path('..') / 'data'
RESULTS_DIR.mkdir(exist_ok=True)
DATA_DIR.mkdir(exist_ok=True)

n_models = sum(len(p['models']) for p in PROVIDERS.values())
print(f"Experiment config:")
print(f"  Providers:  {len(PROVIDERS)}")
print(f"  Models:     {n_models}")
print(f"  Runs/model: {RUNS_PER_MODEL}")
print(f"  Expected:   {n_models * RUNS_PER_MODEL * 3} persona rows ({n_models * RUNS_PER_MODEL} workflows)")

Experiment config:
  Providers:  5
  Models:     14
  Runs/model: 22
  Expected:   924 persona rows (308 workflows)


## 2. API Client

All 5 providers expose OpenAI-compatible chat completions APIs,
so we use the `openai` Python library as a unified client.

In [2]:
def get_client(provider_name: str) -> OpenAI:
    """Create an OpenAI-compatible client for the given provider."""
    provider = PROVIDERS[provider_name]
    api_key = API_KEYS[provider_name]
    if not api_key:
        raise ValueError(f"API key for '{provider_name}' is empty. Fill it in config.py.")
    return OpenAI(base_url=provider["base_url"], api_key=api_key, timeout=60.0)


def call_llm(client: OpenAI, model_id: str, messages: list, params: dict = None,
             max_retries: int = 3) -> str:
    """Send a chat completion request with retry logic. Returns response text."""
    if params is None:
        params = GENERATION_PARAMS

    for attempt in range(max_retries):
        try:
            kwargs = dict(
                model=model_id,
                messages=messages,
                temperature=params.get("temperature", 0.7),
                max_tokens=params.get("max_tokens", 2048),
            )
            if kwargs["temperature"] > 0:
                kwargs["top_p"] = params.get("top_p", 0.9)
            response = client.chat.completions.create(**kwargs)
            text = response.choices[0].message.content
            text = re.sub(r'<think>.*?</think>', '', text, flags=re.DOTALL).strip()
            text = re.sub(r'<think>.*$', '', text, flags=re.DOTALL).strip()
            return text
        except Exception as e:
            err_str = str(e)
            # Don't retry on permanent or rate-limit errors
            is_permanent = any(code in err_str for code in ["400", "401", "402", "403", "404", "429", "decommissioned"])
            if is_permanent or attempt >= max_retries - 1:
                raise
            wait = 2 ** attempt * 5
            print(f"    Retry {attempt+1}: {e}. Waiting {wait}s...")
            time.sleep(wait)


def run_prompt_workflow(client: OpenAI, model_id: str, delay: float) -> dict:
    """Execute Prompt 1 -> Prompt 2 workflow. Returns raw response texts."""
    messages_p1 = [{"role": "user", "content": PROMPT_1}]
    response_p1 = call_llm(client, model_id, messages_p1)
    time.sleep(delay)

    messages_p2 = [
        {"role": "user", "content": PROMPT_1},
        {"role": "assistant", "content": response_p1},
        {"role": "user", "content": PROMPT_2},
    ]
    response_p2 = call_llm(client, model_id, messages_p2)
    time.sleep(delay)

    return {"prompt1_response": response_p1, "prompt2_response": response_p2}

## 3. Connectivity Test

Verify each provider's API key and model access before starting the full collection.

In [3]:
print("Testing provider connections...\n")
active_providers = []

for pname, pcfg in PROVIDERS.items():
    if not API_KEYS.get(pname):
        print(f"  [{pname}] SKIPPED — no API key")
        continue
    try:
        client = get_client(pname)
        model_id = pcfg["models"][0]["id"]
        test = call_llm(client, model_id,
                        [{"role": "user", "content": "Say hello in one word."}],
                        {"temperature": 0, "max_tokens": 10})
        print(f"  [{pname}] OK — {pcfg['models'][0]['name']}: \"{test[:40]}\"")
        active_providers.append(pname)
    except Exception as e:
        print(f"  [{pname}] FAILED — {e}")

active_models = sum(len(PROVIDERS[p]["models"]) for p in active_providers)
print(f"\nActive: {len(active_providers)} providers, {active_models} models")
print(f"Expected workflows: {active_models * RUNS_PER_MODEL}")
print(f"Expected persona rows: {active_models * RUNS_PER_MODEL * 3}")

Testing provider connections...

  [groq] OK — llama-3.1-8b: "Hello."
  [mistral] OK — mistral-small: "Hi!"
  [google_ai] OK — gemma-3-4b: "Hello!"
  [sambanova] FAILED — Error code: 429 - {'error': {'code': None, 'message': 'Rate limit exceeded', 'param': None, 'type': 'rate_limit_exceeded'}}
  [cerebras] OK — llama3.1-8b: "Hello."

Active: 4 providers, 11 models
Expected workflows: 242
Expected persona rows: 726


## 4. Collect Raw Responses

For each provider/model, run Prompt 1 → Prompt 2 workflow `RUNS_PER_MODEL` times.
Results are saved incrementally to `results/raw_responses.json` for resume support.

In [4]:
RAW_FILE = Path("..") / "data" / "raw_responses.json"

# Load existing results if resuming
if RAW_FILE.exists():
    with open(RAW_FILE, "r", encoding="utf-8") as f:
        raw_results = json.load(f)
    print(f"Resuming: {len(raw_results)} existing results loaded.")
else:
    raw_results = []

completed = {(r["provider"], r["model_name"], r["run"]) for r in raw_results}


def save_raw():
    with open(RAW_FILE, "w", encoding="utf-8") as f:
        json.dump(raw_results, f, ensure_ascii=False, indent=2)


# Count remaining work
total_tasks = sum(
    len(PROVIDERS[p]["models"]) * RUNS_PER_MODEL
    for p in active_providers
)
already_done = sum(1 for p in active_providers
                   for m in PROVIDERS[p]["models"]
                   for r in range(1, RUNS_PER_MODEL + 1)
                   if (p, m["name"], r) in completed)

print(f"Total tasks: {total_tasks}, Already done: {already_done}, Remaining: {total_tasks - already_done}")

Resuming: 285 existing results loaded.
Total tasks: 242, Already done: 240, Remaining: 2


In [5]:
errors = []
skipped_models = set()

for provider_name in active_providers:
    provider_cfg = PROVIDERS[provider_name]
    client = get_client(provider_name)
    delay = REQUEST_DELAY.get(provider_name, 2.0)

    for model_cfg in provider_cfg["models"]:
        model_name = model_cfg["name"]
        model_id = model_cfg["id"]
        consecutive_fails = 0
        success_count = sum(1 for r in raw_results if r["provider"] == provider_name and r["model_name"] == model_name)

        print(f"\n--- {provider_name}/{model_name} (done: {success_count}/{RUNS_PER_MODEL}) ---")

        for run in range(1, RUNS_PER_MODEL + 1):
            if (provider_name, model_name, run) in completed:
                continue

            if consecutive_fails >= 3:
                skipped_models.add(f"{provider_name}/{model_name}")
                break  # Skip remaining runs for this model

            print(f"  run {run}...", end=" ", flush=True)

            try:
                result = run_prompt_workflow(client, model_id, delay)
                raw_results.append({
                    "provider": provider_name,
                    "model_name": model_name,
                    "model_id": model_id,
                    "run": run,
                    "timestamp": datetime.now().isoformat(),
                    "prompt1_response": result["prompt1_response"],
                    "prompt2_response": result["prompt2_response"],
                })
                completed.add((provider_name, model_name, run))
                consecutive_fails = 0
                success_count += 1
                print(f"OK ({success_count}/{RUNS_PER_MODEL})")

                if len(raw_results) % 5 == 0:
                    save_raw()

            except Exception as e:
                consecutive_fails += 1
                err_msg = f"{provider_name}/{model_name} run {run}: {e}"
                errors.append(err_msg)
                print(f"FAIL ({str(e)[:80]})")
                time.sleep(2)

save_raw()

print(f"\n{'='*60}")
print(f"Collection complete: {len(raw_results)} workflows saved.")
if skipped_models:
    print(f"Skipped models (3+ failures): {skipped_models}")
if errors:
    print(f"Errors: {len(errors)}")
    for e in errors[:10]:
        print(f"  - {e[:150]}")


--- groq/llama-3.1-8b (done: 25/22) ---

--- groq/llama-3.3-70b (done: 25/22) ---

--- groq/llama-4-scout (done: 25/22) ---

--- mistral/mistral-small (done: 25/22) ---

--- mistral/mistral-nemo (done: 25/22) ---

--- mistral/ministral-8b (done: 25/22) ---

--- google_ai/gemma-3-4b (done: 24/22) ---

--- google_ai/gemma-3-12b (done: 1/22) ---
  run 2... OK (2/22)
  run 3...     Retry 1: Error code: 503 - [{'error': {'code': 503, 'message': 'This model is currently experiencing high demand. Spikes in demand are usually temporary. Please try again later.', 'status': 'UNAVAILABLE'}}]. Waiting 5s...
OK (3/22)
  run 4... OK (4/22)
  run 5... OK (5/22)
  run 6... OK (6/22)
  run 7... OK (7/22)
  run 8... OK (8/22)
  run 9... OK (9/22)
  run 10... OK (10/22)
  run 11... OK (11/22)
  run 12... OK (12/22)
  run 13... OK (13/22)
  run 14... OK (14/22)
  run 15...     Retry 1: Error code: 503 - [{'error': {'code': 503, 'message': 'This model is currently experiencing high demand. Spikes in deman

## 5. Parse Responses into Structured Dataset

Use Groq Llama 3.3 70B (temperature=0) to extract structured persona fields from
the free-form responses. Each workflow yields 3 persona rows (1 vulnerable, 2 not).

In [5]:
# Reload raw responses
with open(RAW_FILE, "r", encoding="utf-8") as f:
    raw_results = json.load(f)
print(f"Loaded {len(raw_results)} raw responses to parse.")

# Parsing client
parse_client = get_client(PARSE_PROVIDER)
parse_delay = REQUEST_DELAY.get(PARSE_PROVIDER, 2.0)


def parse_response(raw: dict) -> list:
    """Parse one raw workflow into 3 structured persona records."""
    prompt = PARSE_PROMPT.format(
        prompt1_response=raw["prompt1_response"],
        prompt2_response=raw["prompt2_response"],
    )
    text = call_llm(parse_client, PARSE_MODEL,
                    [{"role": "user", "content": prompt}],
                    PARSE_PARAMS)
    time.sleep(parse_delay)

    # Clean markdown fences if present
    text = text.strip()
    if text.startswith("```"):
        text = re.sub(r'^```(?:json)?\s*', '', text)
        text = re.sub(r'\s*```$', '', text)

    personas = json.loads(text)
    if not isinstance(personas, list) or len(personas) != 3:
        raise ValueError(f"Expected 3 personas, got {len(personas) if isinstance(personas, list) else type(personas)}")

    # Attach metadata
    for i, p in enumerate(personas):
        p["provider"] = raw["provider"]
        p["model_name"] = raw["model_name"]
        p["model_id"] = raw["model_id"]
        p["run"] = raw["run"]
        p["persona_id"] = f"P{i+1}"
        p["prompt1_response"] = raw["prompt1_response"]
        p["prompt2_response"] = raw["prompt2_response"]

    return personas

Loaded 285 raw responses to parse.


In [6]:
PARSED_FILE = Path("..") / "data" / "parsed_personas.json"

# Resume support
if PARSED_FILE.exists():
    with open(PARSED_FILE, "r", encoding="utf-8") as f:
        all_personas = json.load(f)
    parsed_keys = {(p["provider"], p["model_name"], p["run"]) for p in all_personas}
    print(f"Resuming: {len(all_personas)} personas already parsed.")
else:
    all_personas = []
    parsed_keys = set()

parse_errors = []

for raw in tqdm(raw_results, desc="Parsing responses"):
    key = (raw["provider"], raw["model_name"], raw["run"])
    if key in parsed_keys:
        continue

    try:
        personas = parse_response(raw)
        all_personas.extend(personas)
        parsed_keys.add(key)

        if len(parsed_keys) % 10 == 0:
            with open(PARSED_FILE, "w", encoding="utf-8") as f:
                json.dump(all_personas, f, ensure_ascii=False, indent=2)

    except Exception as e:
        parse_errors.append(f"{key}: {e}")

# Final save
with open(PARSED_FILE, "w", encoding="utf-8") as f:
    json.dump(all_personas, f, ensure_ascii=False, indent=2)

print(f"\nParsed: {len(all_personas)} persona records from {len(parsed_keys)} workflows.")
if parse_errors:
    print(f"Parse errors ({len(parse_errors)}):")
    for e in parse_errors[:10]:
        print(f"  - {e}")

Resuming: 135 personas already parsed.


Parsing responses:   0%|          | 0/285 [00:00<?, ?it/s]


Parsed: 855 persona records from 285 workflows.


## 6. Build Final Dataset (CSV)

Convert parsed personas into the structured dataset format from Week 5 slides (p.151):
Model | Persona ID | Name | Age | Gender | Education | Personality Traits | Domain | Experience | Location | Devices | Is Vulnerable | Reasons

In [7]:
with open(PARSED_FILE, "r", encoding="utf-8") as f:
    all_personas = json.load(f)

rows = []
for p in all_personas:
    rows.append({
        "Provider": p.get("provider", ""),
        "Model": p.get("model_name", ""),
        "Model_ID": p.get("model_id", ""),
        "Run": p.get("run", ""),
        "Persona_ID": p.get("persona_id", ""),
        "Name": p.get("name", ""),
        "Age": p.get("age", ""),
        "Gender": p.get("gender", ""),
        "Education_Level": p.get("education_level", ""),
        "Personality_Traits": p.get("personality_traits", ""),
        "Domain_of_Work": p.get("domain_of_work", ""),
        "Years_of_Experience": p.get("years_of_experience", ""),
        "Location": p.get("location", ""),
        "Devices_and_Technologies": p.get("devices_and_technologies", ""),
        "Is_Vulnerable": "Yes" if p.get("is_vulnerable") else "No",
        "Vulnerability_Reasons": p.get("vulnerability_reasons", ""),
        "Prompt1_Response": p.get("prompt1_response", ""),
        "Prompt2_Response": p.get("prompt2_response", ""),
    })

df = pd.DataFrame(rows)

DATASET_FILE = RESULTS_DIR / "dataset.csv"
df.to_csv(DATASET_FILE, index=False, encoding="utf-8-sig")

print(f"Dataset saved: {DATASET_FILE}")
print(f"Total rows: {len(df)}")
print(f"\nBreakdown by provider and model:")
print(df.groupby(["Provider", "Model"]).size().to_string())
print(f"\nVulnerable: {(df['Is_Vulnerable'] == 'Yes').sum()} / {len(df)}")

Dataset saved: results\dataset.csv
Total rows: 855

Breakdown by provider and model:
Provider   Model           
cerebras   llama3.1-8b         66
           qwen3-235b          66
google_ai  gemma-3-12b         60
           gemma-3-27b         66
           gemma-3-4b          72
groq       llama-3.1-8b        75
           llama-3.3-70b       75
           llama-4-scout       75
mistral    ministral-8b        75
           mistral-nemo        75
           mistral-small       75
sambanova  deepseek-v3         27
           llama-3.3-70b       21
           llama-4-maverick    27

Vulnerable: 285 / 855


## 7. Data Quality Check

In [8]:
print("=== Data Quality Check ===\n")

# Gender distribution
print("Gender distribution (all personas):")
print(df["Gender"].value_counts())

# Age statistics
df["Age"] = pd.to_numeric(df["Age"], errors="coerce")
print(f"\nAge statistics:")
print(df["Age"].describe().round(1))

# Vulnerable per run should be exactly 1 per 3 rows
vuln_per_run = df[df["Is_Vulnerable"] == "Yes"].groupby(["Provider", "Model", "Run"]).size()
print(f"\nVulnerable per run (expect 1.0):")
print(f"  Mean: {vuln_per_run.mean():.2f}, Min: {vuln_per_run.min()}, Max: {vuln_per_run.max()}")

# Top domains
print(f"\nTop 10 domains of work:")
print(df["Domain_of_Work"].value_counts().head(10))

# Top locations
print(f"\nTop 10 locations:")
print(df["Location"].value_counts().head(10))

=== Data Quality Check ===

Gender distribution (all personas):
Gender
Female                 391
Male                   277
Non-binary             160
She/Her                 13
He/Him                   7
Non-Binary               2
Genderfluid              2
Non-binary friendly      1
N/A                      1
They/Them                1
Name: count, dtype: int64

Age statistics:
count    855.0
mean      32.5
std       10.3
min        0.0
25%       24.0
50%       28.0
75%       42.0
max       68.0
Name: Age, dtype: float64

Vulnerable per run (expect 1.0):
  Mean: 1.00, Min: 1, Max: 1

Top 10 domains of work:
Domain_of_Work
Marketing                              27
Sustainability Consultant              23
Software Development                   18
Healthcare                             17
Data Analysis & Predictive Modeling    16
Education                              15
Artificial Intelligence                13
Tech/Data Analytics                    13
Engineering                    

## 8. Quick Summary — Vulnerability Selection Patterns

In [9]:
vuln = df[df["Is_Vulnerable"] == "Yes"]
non_vuln = df[df["Is_Vulnerable"] == "No"]

print("=== Vulnerability Selection Summary ===\n")

print("Gender of vulnerable personas:")
print(vuln["Gender"].value_counts())
print(f"\nGender of non-vulnerable personas:")
print(non_vuln["Gender"].value_counts())

print(f"\nMean age — Vulnerable: {vuln['Age'].mean():.1f}, Non-vulnerable: {non_vuln['Age'].mean():.1f}")

print(f"\nTop 5 domains chosen as vulnerable:")
print(vuln["Domain_of_Work"].value_counts().head(5))

print(f"\nTop 5 education levels chosen as vulnerable:")
print(vuln["Education_Level"].value_counts().head(5))

print(f"\nTop 5 locations chosen as vulnerable:")
print(vuln["Location"].value_counts().head(5))

=== Vulnerability Selection Summary ===

Gender of vulnerable personas:
Gender
Female                 112
Non-binary             111
Male                    51
She/Her                  5
Non-binary friendly      1
Non-Binary               1
Genderfluid              1
N/A                      1
He/Him                   1
They/Them                1
Name: count, dtype: int64

Gender of non-vulnerable personas:
Gender
Female         279
Male           226
Non-binary      49
She/Her          8
He/Him           6
Non-Binary       1
Genderfluid      1
Name: count, dtype: int64

Mean age — Vulnerable: 28.7, Non-vulnerable: 34.4

Top 5 domains chosen as vulnerable:
Domain_of_Work
Marketing            9
Education            8
Arts                 7
Art                  6
Retail/Management    5
Name: count, dtype: int64

Top 5 education levels chosen as vulnerable:
Education_Level
Bachelor's               59
Bachelor's (pursuing)    22
Diploma                  22
Master's                 22
High 